# 02 EEA Batch Ingestion

**Phase:** 3 — EEA Batch Ingestion  
**Status:** Issue 3.1 complete (source access and data policy documented). Issues 3.2–3.8 pending.

This notebook is the readable Phase 3 documentation trail for EEA historical
air quality batch ingestion. It documents source access, raw-data policy,
schema contracts, data quality rules, and output conventions phase by phase
as later issues are resolved.

---

## Phase 3 Scope

Phase 3 implements controlled EEA historical air quality batch ingestion.
It is limited to the **8 starter cities** and the **3 core pollutants**
(PM2.5, PM10, NO2) defined in Phase 2.

Phase 3 **must not** implement:

- Wikipedia scraping
- Open-Meteo API client behaviour
- Kafka producer logic
- Spark Structured Streaming
- Gold tables
- Dashboards, Airflow, dbt, PostgreSQL, cloud deployment, or ML

Historical EEA batch data is **not** the same as Open-Meteo live/current API
data. These two data contexts must remain separated throughout the pipeline.
Silver and Gold tables must use a `data_context` or `source` field to make
this distinction explicit (`historical_eea` vs `live_open_meteo`).

---

## Phase 3 EEA Source Access (Issue 3.1)

This section documents how EEA raw data is obtained, stored locally, and
kept out of the repository. It is documentation and hygiene only. No data
download, loader implementation, Spark job, or Silver/Gold output belongs
here.

Full policy details are in `docs/data_sources.md` under the
**Phase 3 EEA Source Access** heading.

---

### EEA Source Access Path

EEA historical air quality time series are available through:

| Access path | URL | Phase 3 use |
| --- | --- | --- |
| EEA Air Quality Download web app | `https://eeadmz1-downloads-webapp.azurewebsites.net` | Select country, station, pollutant, year range; download Parquet or CSV. |
| EEA station spatial service | ArcGIS REST layer at `air.discomap.eea.europa.eu` | Re-query for specific station EoI codes identified in Phase 1. |
| Station-specific Parquet links | Embedded in EEA station popup metadata | Preferred access path for per-station, per-pollutant E1a validated time series. |

Phase 1 already identified candidate stations for Vienna (`AT90TAB`, `AT90AKC`)
and Berlin (`DEBE068`). Phase 3 must extend station review to all 8 starter
cities before ingestion logic is written (Issue 3.3).

---

### Raw-Data Policy

| Rule | Detail |
| --- | --- |
| Large raw EEA files **must not** be committed | `data/**/*.parquet`, `data/**/*.csv`, `data/**/*.json` are git-ignored. |
| Raw files live under `data/bronze/eea/` | This directory is git-ignored. `.gitkeep` maintains the folder structure. |
| Raw files must be reproducibly referenced | Document the station ID, pollutant, year range, and download endpoint so any reviewer can re-download the same files. |
| Tiny test fixtures are allowed | Small in-memory or temporary pytest fixtures only; not real bulk data; not committed. |
| No bulk downloads | Only the stations and time periods needed for 8 cities and 3 pollutants. |

---

### Naming Convention For Local EEA Files

Files stored under `data/bronze/eea/` must follow this pattern:

```
eea_<station_id>_<pollutant_key>_<year_start>_<year_end>.<ext>
```

Examples:

```
eea_AT90TAB_pm25_2018_2023.parquet
eea_AT90TAB_no2_2018_2023.parquet
eea_DEBE068_pm10_2020_2024.csv
```

Where `<pollutant_key>` is one of `pm25`, `pm10`, `no2`.  
Tiny local validation samples may use a `sample_` prefix:

```
sample_eea_AT90TAB_pm25_2022.csv
```

---

### Git-Ignore Verification

The repository `.gitignore` protects all EEA data files:

```gitignore
data/**/*.parquet
data/**/*.csv
data/**/*.json
data/**/*.html
data/**/checkpoints/**
!**/.gitkeep
```

Verify a specific file is ignored before attempting to add it:

```bash
git check-ignore -v data/bronze/eea/sample_test.csv
```

Expected output: a line referencing `.gitignore` and the file path. If the
file is not ignored, do not proceed until the `.gitignore` rules are fixed.

---

### Reproducibility Contract

Because large EEA raw files are not committed, the following information must
be recorded in `docs/data_sources.md` for every EEA download used in Phase 3:

| Item | Example value |
| --- | --- |
| Station EoI code | `AT90TAB` |
| Station name | `Taborstraße` |
| City reference `city_id` | `vienna_at` |
| Pollutant | PM2.5 |
| Year range | 2018–2023 |
| Download URL or endpoint | `https://eeadmz1-downloads-webapp.azurewebsites.net` |
| File format | Parquet (E1a validated) |
| Download date | 2026-05-30 |
| Local path | `data/bronze/eea/eea_AT90TAB_pm25_2018_2023.parquet` |

This table will be populated during Issue 3.3 (station-to-city mapping) and
Issue 3.4 (EEA loader implementation).

---

### Issue 3.1 Definition Of Done

- [x] EEA source access path is documented.
- [x] Raw-data policy for EEA files is documented.
- [x] Naming convention for local EEA samples is documented.
- [x] `.gitignore` behaviour is confirmed (all `data/**/*.csv`, `*.parquet`, `*.json` are ignored).
- [x] Scope boundary is explicit: no bulk download, no loader, no Spark, no Gold output.
- [x] `docs/data_sources.md` contains `Phase 3 EEA Source Access` section.
- [x] This notebook contains a matching Markdown summary.

---

## Phase 3 Pending Work (Issues 3.2–3.8)

The following sections will be added to this notebook as later Phase 3 issues
are resolved:

| Issue | Title | Status |
| --- | --- | --- |
| 3.2 | Define EEA input schema and Silver output schema | pending |
| 3.3 | Prepare EEA station-to-city mapping table | pending |
| 3.4 | Implement EEA loader for controlled local files | pending |
| 3.5 | Add EEA data quality validation rules | pending |
| 3.6 | Build EEA city daily Silver Parquet | pending |
| 3.7 | Update this notebook with full Phase 3 documentation | pending |
| 3.8 | Phase 3 QA report and gate decision | pending |

---

## Phase 3 Deliverables

| Artefact | Path | Status |
| --- | --- | --- |
| This notebook | `notebooks/02_eea_batch_ingestion.ipynb` | Issue 3.1 content added |
| Source access documentation | `docs/data_sources.md` | Issue 3.1 section added |
| EEA Loader | `src/ingestion/eea_loader.py` | pending (Issue 3.4) |
| Batch Processing Job | `src/spark_jobs/batch_eea_processing_job.py` | pending (Issue 3.4) |
| Silver Output | `data/silver/eea_city_daily.parquet` | pending (Issue 3.6) |
| Data Quality Summary | `docs/data_sources.md` or this notebook | pending (Issue 3.5) |
| Phase 3 QA Report | `docs/qa/phase3_qa_report.md` | pending (Issue 3.8) |